<a href="https://colab.research.google.com/github/Sambca-hub/python-project-camous-store-and-billing-system/blob/main/Project_3_CampusStoreandBillingSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
class CampusStore:
    def __init__(self):
        self.inventory = {}

        self.cart = []

        self.tax_rate = 0.05  # 5% tax
        self.discount_slabs = {
            1000: 0.05,  # 5% discount for subtotal > 1000
            2500: 0.10   # 10% discount for subtotal > 2500
        }
        self.default_low_stock_threshold = 10

    def _check_low_stock(self, product_name):
        if product_name in self.inventory:
            stock = self.inventory[product_name]['stock']
            threshold = self.inventory[product_name].get('low_stock_threshold', self.default_low_stock_threshold)
            if stock <= threshold:
                print(f"[ALERT] Low stock for {product_name}: {stock} units left!")

    def add_product(self, product_name, price, stock, low_stock_threshold=None):
        if not isinstance(product_name, str) or not product_name:
            print("Error: Product name must be a non-empty string.")
            return False
        if not isinstance(price, (int, float)) or price <= 0:
            print("Error: Price must be a positive number.")
            return False
        if not isinstance(stock, int) or stock < 0:
            print("Error: Stock must be a non-negative integer.")
            return False

        if product_name in self.inventory:
            print(f"Product '{product_name}' already exists. Use update_stock to modify.")
            return False

        self.inventory[product_name] = {
            'price': price,
            'stock': stock,
            'low_stock_threshold': low_stock_threshold if low_stock_threshold is not None else self.default_low_stock_threshold
        }
        print(f"Product '{product_name}' added with price {price:.2f} and stock {stock}.")
        self._check_low_stock(product_name)
        return True

    def update_stock(self, product_name, quantity_change):
        if product_name not in self.inventory:
            print(f"Error: Product '{product_name}' not found.")
            return False
        if not isinstance(quantity_change, int) or quantity_change == 0:
            print("Error: Quantity change must be a non-zero integer.")
            return False

        current_stock = self.inventory[product_name]['stock']
        new_stock = current_stock + quantity_change

        if new_stock < 0:
            print(f"Error: Cannot reduce stock of '{product_name}' to a negative value. Current stock: {current_stock}.")
            return False

        self.inventory[product_name]['stock'] = new_stock
        print(f"Stock for '{product_name}' updated from {current_stock} to {new_stock}.")
        self._check_low_stock(product_name)
        return True

    def update_price(self, product_name, new_price):
        if product_name not in self.inventory:
            print(f"Error: Product '{product_name}' not found.")
            return False
        if not isinstance(new_price, (int, float)) or new_price <= 0:
            print("Error: New price must be a positive number.")
            return False

        old_price = self.inventory[product_name]['price']
        self.inventory[product_name]['price'] = new_price
        print(f"Price for '{product_name}' updated from {old_price:.2f} to {new_price:.2f}.")
        return True

    def display_inventory(self):
        if not self.inventory:
            print("Inventory is empty.")
            return
        print("\n--- Current Inventory ---")
        print(f"{'Product':<20}{'Price':<10}{'Stock':<10}{'Threshold':<12}")
        print("-" * 52)
        for product, details in self.inventory.items():
            print(f"{product:<20}{details['price']:<10.2f}{details['stock']:<10}{details['low_stock_threshold']:<12}")
        print("-------------------------")

    def add_to_cart(self, product_name, quantity):
        if product_name not in self.inventory:
            print(f"Error: Product '{product_name}' not found in inventory.")
            return False
        if not isinstance(quantity, int) or quantity <= 0:
            print("Error: Quantity must be a positive integer.")
            return False

        available_stock = self.inventory[product_name]['stock']
        if quantity > available_stock:
            print(f"Error: Not enough stock for '{product_name}'. Available: {available_stock}, Requested: {quantity}.")
            return False


        for item in self.cart:
            if item['product_name'] == product_name:
                item['quantity'] += quantity
                print(f"{quantity} units of '{product_name}' added to cart. Total in cart: {item['quantity']}.")
                return True

        price = self.inventory[product_name]['price']
        self.cart.append({'product_name': product_name, 'quantity': quantity, 'price': price})
        print(f"{quantity} units of '{product_name}' added to cart.")
        return True

    def view_cart(self):
        if not self.cart:
            print("Cart is empty.")
            return
        print("\n--- Your Shopping Cart ---")
        for item in self.cart:
            print(f"{item['product_name']:<20} x {item['quantity']:<3} @ {item['price']:.2f} = {item['quantity'] * item['price']:.2f}")
        print("--------------------------")

    def generate_bill(self):
        if not self.cart:
            print("Cannot generate bill: Cart is empty.")
            return None

        print("\n--- Campus Store Bill ---")
        print(f"{'Item':<20}{'Qty':<5}{'Price':<8}{'Total':<10}")
        print("-" * 43)

        subtotal = 0.0
        items_for_bill = []

        for item in self.cart:
            product_name = item['product_name']
            quantity = item['quantity']
            price = item['price']
            item_total = quantity * price
            subtotal += item_total
            items_for_bill.append((product_name, quantity, price, item_total))
            print(f"{product_name:<20}{quantity:<5}{price:<8.2f}{item_total:<10.2f}")


            self.update_stock(product_name, -quantity)

        print("-" * 43)
        print(f"{'Subtotal:':<33}{subtotal:<10.2f}")


        discount_percentage = 0.0

        sorted_slabs = sorted(self.discount_slabs.items(), key=lambda x: x[0], reverse=True)
        for threshold, discount_rate in sorted_slabs:
            if subtotal >= threshold:
                discount_percentage = discount_rate
                break

        discount_amount = subtotal * discount_percentage
        if discount_amount > 0:
            print(f"{'Discount ({int(discount_percentage * 100)}%):':<33}-{discount_amount:<10.2f}")

        amount_after_discount = subtotal - discount_amount
        tax_amount = amount_after_discount * self.tax_rate
        final_payable_amount = amount_after_discount + tax_amount

        print(f"{'Tax ({int(self.tax_rate * 100)}%):':<33}{tax_amount:<10.2f}")
        print("-" * 43)
        print(f"{'Final Payable Amount:':<33}{final_payable_amount:<10.2f}")
        print("-------------------------")

        self.cart = []
        return final_payable_amount

    def run_admin_menu(self):
        while True:
            print("\n--- Admin Menu ---")
            print("1. Add New Product")
            print("2. Update Product Stock")
            print("3. Update Product Price")
            print("4. View Inventory")
            print("5. Back to Main Menu")
            choice = input("Enter your choice: ")

            if choice == '1':
                name = input("Enter product name: ").strip()
                try:
                    price = float(input("Enter price: "))
                    stock = int(input("Enter initial stock: "))
                    threshold_input = input(f"Enter low stock threshold (default {self.default_low_stock_threshold}): ").strip()
                    threshold = int(threshold_input) if threshold_input else None
                    self.add_product(name, price, stock, threshold)
                except ValueError:
                    print("Invalid input for price, stock, or threshold. Please enter numbers.")
            elif choice == '2':
                name = input("Enter product name to update stock: ").strip()
                try:
                    quantity_change = int(input("Enter quantity change (e.g., 5 for add, -3 for remove): "))
                    self.update_stock(name, quantity_change)
                except ValueError:
                    print("Invalid quantity change. Please enter an integer.")
            elif choice == '3':
                name = input("Enter product name to update price: ").strip()
                try:
                    new_price = float(input("Enter new price: "))
                    self.update_price(name, new_price)
                except ValueError:
                    print("Invalid price. Please enter a number.")
            elif choice == '4':
                self.display_inventory()
            elif choice == '5':
                break
            else:
                print("Invalid choice. Please try again.")

    def run_customer_menu(self):
        while True:
            print("\n--- Customer Menu ---")
            print("1. View Available Products")
            print("2. Add Product to Cart")
            print("3. View Cart")
            print("4. Checkout (Generate Bill)")
            print("5. Back to Main Menu")
            choice = input("Enter your choice: ")

            if choice == '1':
                self.display_inventory()
            elif choice == '2':
                name = input("Enter product name to add to cart: ").strip()
                try:
                    quantity = int(input("Enter quantity: "))
                    self.add_to_cart(name, quantity)
                except ValueError:
                    print("Invalid quantity. Please enter an integer.")
            elif choice == '3':
                self.view_cart()
            elif choice == '4':
                self.generate_bill()
            elif choice == '5':
                break
            else:
                print("Invalid choice. Please try again.")

    def run(self):
        print("Welcome to Campus Store Inventory & Billing System!")
        while True:
            print("\n--- Main Menu ---")
            print("1. Admin Login")
            print("2. Customer Shopping")
            print("3. Exit")
            main_choice = input("Enter your choice: ")

            if main_choice == '1':
                self.run_admin_menu()
            elif main_choice == '2':
                self.run_customer_menu()
            elif main_choice == '3':
                print("Thank you for using the Campus Store system. Goodbye!")
                break
            else:
                print("Invalid choice. Please try again.")



if __name__ == '__main__':
    store = CampusStore()


    store.add_product("Notebook", 25.00, 100)
    store.add_product("Pen Set", 10.50, 50, 5)
    store.add_product("Water Bottle", 75.00, 30)
    store.add_product("Backpack", 500.00, 20)
    store.add_product("Textbook: Python", 1200.00, 15)

    store.run()


Product 'Notebook' added with price 25.00 and stock 100.
Product 'Pen Set' added with price 10.50 and stock 50.
Product 'Water Bottle' added with price 75.00 and stock 30.
Product 'Backpack' added with price 500.00 and stock 20.
Product 'Textbook: Python' added with price 1200.00 and stock 15.
Welcome to Campus Store Inventory & Billing System!

--- Main Menu ---
1. Admin Login
2. Customer Shopping
3. Exit
Enter your choice: 2

--- Customer Menu ---
1. View Available Products
2. Add Product to Cart
3. View Cart
4. Checkout (Generate Bill)
5. Back to Main Menu
Enter your choice: 2
Enter product name to add to cart: Notebook
Enter quantity: 3
3 units of 'Notebook' added to cart.

--- Customer Menu ---
1. View Available Products
2. Add Product to Cart
3. View Cart
4. Checkout (Generate Bill)
5. Back to Main Menu
Enter your choice: 4

--- Campus Store Bill ---
Item                Qty  Price   Total     
-------------------------------------------
Notebook            3    25.00   75.00     